# 06b – Thử cải tiến TF-IDF (P4)

Câu hỏi: đổi cách biểu diễn văn bản hoặc trọng số nguồn dữ liệu có làm mô hình tốt hơn trên Shopee không?

**Quy tắc chọn:** cấu hình được chọn bằng GroupKFold 5 fold trên B (mỗi fold train trên A + phần còn lại của B, kiểm tra trên sản phẩm chưa thấy, nhãn sao). **Tập vàng không tham gia chọn**, chỉ chấm một lần ở cuối.

Hạn chế: điểm CV dùng nhãn sao, vì chưa có tập Shopee nhãn tay riêng để làm validation.

Thời gian chạy: khoảng 10 phút.

In [1]:
import sys, time, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from joblib import Parallel, delayed
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, precision_score
from sklearn.model_selection import GroupKFold
from sklearn.svm import LinearSVC
from src.data import load_sets, load_gold
from src.preprocess import normalize_text

S = load_sets()
gold, temp = load_gold()
assert not temp
for k in ["A", "A_dev", "A_test", "B", "Aug"]:
    S[k] = S[k].assign(text_norm=S[k].text_raw.map(normalize_text))
gold = gold.assign(text_norm=gold.text_raw.map(normalize_text))

## 1. Các hàm dùng chung

`word_prep`: từ đã tách; `word_norm`: chuẩn hóa, chưa tách từ; `char`: `char_wb` 3–5 ký tự. `wa`, `wb`, `aug` là trọng số mẫu của A, B và Aug (0 = không dùng).

In [2]:
class Rep:
    """Biểu diễn văn bản; ghép nhiều TF-IDF nếu cần."""
    def __init__(self, name):
        self.name = name
        word = lambda: TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=1)
        char = lambda: TfidfVectorizer(sublinear_tf=True, analyzer="char_wb", ngram_range=(3, 5), min_df=2)
        self.parts = {"word_prep": [("text_prep", word())], "word_norm": [("text_norm", word())],
                      "char": [("text_norm", char())],
                      "word_norm+char": [("text_norm", word()), ("text_norm", char())],
                      "word_prep+char": [("text_prep", word()), ("text_norm", char())]}[name]

    def fit_transform(self, d):
        return hstack([v.fit_transform(d[c]) for c, v in self.parts]).tocsr()

    def transform(self, d):
        return hstack([v.transform(d[c]) for c, v in self.parts]).tocsr()


def make_clf(kind, C):
    if kind == "LR":
        return LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
    return LinearSVC(C=C, class_weight="balanced")


def train_data(B_part, wa, wb, aug):
    parts, w = [S["A"], B_part], [np.full(len(S["A"]), wa), np.full(len(B_part), wb)]
    if aug:
        parts.append(S["Aug"]); w.append(np.full(len(S["Aug"]), aug))
    return pd.concat(parts, ignore_index=True), np.concatenate(w)


def fit(rep_name, kind, C, B_part, wa, wb, aug):
    d, w = train_data(B_part, wa, wb, aug)
    r = Rep(rep_name)
    X = r.fit_transform(d)
    return r, make_clf(kind, C).fit(X, d.label, sample_weight=w)


def one_fold(cfg, tr, va):
    r, m = fit(*cfg[:3], S["B"].iloc[tr], *cfg[3:])
    v = S["B"].iloc[va]
    p = m.predict(r.transform(v))
    return f1_score(v.label, p, average="macro"), recall_score(v.label, p, labels=["neg"], average="macro")


def cv_score(cfg):
    folds = list(GroupKFold(5).split(S["B"], groups=S["B"].group))
    rs = Parallel(n_jobs=5)(delayed(one_fold)(cfg, tr, va) for tr, va in folds)
    f = np.array(rs)
    r, m = fit(*cfg[:3], S["B"], *cfg[3:])
    dev = f1_score(S["A_dev"].label, m.predict(r.transform(S["A_dev"])), average="macro")
    return dict(rep=cfg[0], clf=cfg[1], C=cfg[2], wa=cfg[3], wb=cfg[4], aug=cfg[5],
                cv_f1=f[:, 0].mean(), cv_std=f[:, 0].std(), cv_rec_neg=f[:, 1].mean(), dev_f1=dev)

## 2. Dò theo 3 chặng

Chặng 1 chọn biểu diễn, chặng 2 chọn trọng số nguồn, chặng 3 chọn C. Mỗi chặng giữ cấu hình tốt nhất theo `cv_f1`.

In [3]:
rows, t0 = [], time.time()
def run(stage, cfgs):
    for c in cfgs:
        x = cv_score(c); x["stage"] = stage; rows.append(x)
        print(stage, c, f"cv={x['cv_f1']:.3f}±{x['cv_std']:.3f} rec={x['cv_rec_neg']:.3f} dev={x['dev_f1']:.3f} t={time.time()-t0:.0f}", flush=True)
    return pd.DataFrame(rows)


DEF_C = {"LR": 3, "SVC": 0.3}
run("1_bieu_dien", [(rp, k, DEF_C[k], 1, 1, 0) for rp in ["word_prep", "word_norm", "char", "word_norm+char", "word_prep+char"]
                    for k in ["LR", "SVC"]])
b = pd.DataFrame(rows).sort_values("cv_f1").iloc[-1]
run("2_nguon", [(b.rep, b.clf, b.C, wa, wb, aug) for wa in [1, 0.5, 0.25] for wb in [1, 2, 4] for aug in [0, 0.5]
                if not (wa == 1 and wb == 1 and aug == 0)])
b = pd.DataFrame(rows).sort_values("cv_f1").iloc[-1]
grid = {"LR": [0.3, 1, 3, 10, 30], "SVC": [0.03, 0.1, 0.3, 1]}[b.clf]
run("3_C", [(b.rep, b.clf, c, b.wa, b.wb, b.aug) for c in grid if c != b.C])
res = pd.DataFrame(rows)
res.to_csv("../report/thu_cai_tien_cv.csv", index=False)
best = res.sort_values("cv_f1").iloc[-1]
print("CHON:", best.to_dict())

1_bieu_dien ('word_prep', 'LR', 3, 1, 1, 0) cv=0.579±0.109 rec=0.917 dev=0.739 t=19


1_bieu_dien ('word_prep', 'SVC', 0.3, 1, 1, 0) cv=0.610±0.028 rec=0.919 dev=0.760 t=21


1_bieu_dien ('word_norm', 'LR', 3, 1, 1, 0) cv=0.587±0.116 rec=0.930 dev=0.746 t=32


1_bieu_dien ('word_norm', 'SVC', 0.3, 1, 1, 0) cv=0.575±0.104 rec=0.925 dev=0.739 t=35


1_bieu_dien ('char', 'LR', 3, 1, 1, 0) cv=0.568±0.115 rec=0.898 dev=0.719 t=44


1_bieu_dien ('char', 'SVC', 0.3, 1, 1, 0) cv=0.575±0.118 rec=0.898 dev=0.744 t=51


... (21 outputs left out)

3_C ('word_prep', 'SVC', 0.03, np.float64(0.25), np.int64(2), np.float64(0.5)) cv=0.632±0.039 rec=0.871 dev=0.669 t=181


3_C ('word_prep', 'SVC', 0.1, np.float64(0.25), np.int64(2), np.float64(0.5)) cv=0.644±0.030 rec=0.925 dev=0.691 t=185


3_C ('word_prep', 'SVC', 1, np.float64(0.25), np.int64(2), np.float64(0.5)) cv=0.625±0.010 rec=0.920 dev=0.726 t=190


CHON: {'rep': 'word_prep', 'clf': 'SVC', 'C': 0.3, 'wa': 0.25, 'wb': 2, 'aug': 0.5, 'cv_f1': 0.6438357414043537, 'cv_std': 0.024771157931208668, 'cv_rec_neg': 0.9261238231264757, 'dev_f1': 0.7171758377364897, 'stage': '2_nguon'}


## 3. Chấm trên tập vàng

Dòng "Chuẩn hóa, không tách từ" chỉ để đối chiếu với notebook 07. Nó không được chọn bằng CV, nên không dùng làm mô hình cuối.

In [4]:
def final(name, cfg):
    r, m = fit(*cfg[:3], S["B"], *cfg[3:])
    p = m.predict(r.transform(gold))
    t = m.predict(r.transform(S["A_test"]))
    return dict(cau_hinh=name, F1_gold=f1_score(gold.label, p, average="macro"),
                Rec_neg=recall_score(gold.label, p, labels=["neg"], average="macro"),
                Pre_neg=precision_score(gold.label, p, labels=["neg"], average="macro"),
                F1_ViSD_test=f1_score(S["A_test"].label, t, average="macro")), p


cfg_best = (best.rep, best.clf, best.C, best.wa, best.wb, best.aug)
out, preds = [], {}
for n, c in [("Hiện tại TF-IDF+LR", ("word_prep", "LR", 3, 1, 1, 0)),
             ("Hiện tại LinearSVC", ("word_prep", "SVC", 0.3, 1, 1, 0)),
             ("Chuẩn hóa, không tách từ + LR", ("word_norm", "LR", 3, 1, 1, 0)),
             ("Được chọn bằng CV", cfg_best)]:
    x, p = final(n, c); out.append(x); preds[n] = p
print(pd.DataFrame(out).round(3).to_string())
pd.DataFrame(out).round(3).to_csv("../report/thu_cai_tien_gold.csv", index=False)

y, rng = gold.label.values, np.random.default_rng(42)
idx = rng.integers(0, len(y), (1000, len(y)))
for n in ["Chuẩn hóa, không tách từ + LR", "Được chọn bằng CV"]:
    a, b_ = preds[n], preds["Hiện tại TF-IDF+LR"]
    d = np.array([f1_score(y[i], a[i], average="macro") - f1_score(y[i], b_[i], average="macro") for i in idx])
    print(f"{n} − hiện tại: {d.mean():+.3f} CI95 [{np.percentile(d, 2.5):+.3f}, {np.percentile(d, 97.5):+.3f}] P(>0)={(d > 0).mean():.2f}")

                        cau_hinh  F1_gold  Rec_neg  Pre_neg  F1_ViSD_test
0             Hiện tại TF-IDF+LR    0.619    0.826    0.833         0.720
1             Hiện tại LinearSVC    0.587    0.810    0.860         0.696
2  Chuẩn hóa, không tách từ + LR    0.637    0.826    0.826         0.713
3              Được chọn bằng CV    0.588    0.818    0.861         0.690


Chuẩn hóa, không tách từ + LR − hiện tại: +0.019 CI95 [-0.011, +0.062] P(>0)=0.82


Được chọn bằng CV − hiện tại: -0.030 CI95 [-0.085, +0.014] P(>0)=0.14


**Nhận xét thí nghiệm lưu trước gán lại nhãn.** Độ lệch chuẩn CV lớn so với chênh lệch trung bình cấu hình. Tăng trọng số Shopee/giảm ViSD4SA có thể giúp miền Shopee nhưng giảm dev ViSD4SA. Notebook chưa chạy lại với nhãn mới, không dùng kết quả lưu ở đây làm kết quả chính. Điểm vàng chỉ thăm dò, không dùng chọn cấu hình.